In [0]:
from pyspark.sql import functions as F #como o spark é feito em scala, uma linguagem baseada em java, importamos essa biblio pyspark para podermos nos comunicar com a engine através do python, e fica mais fácil e versátil. já estamos instanciando a capacidade de sql do pyspark pra podermos usar comandos sql

# pra trabalhar com o pyspark, podemos usar as funções do pyspark ou sql 
CAMINHO = "/Volumes/voebem/bronze/arquivos/vra/*.csv" #o "*" é pra pegar tudo que tem .csv, vai considerar como fonte
TABELA = "voebem.bronze.vra"

#obs: códigos só vão acontecer quando tivermos uma action


In [0]:
bruto = (
    spark.read.format("csv")
    .option("sep", ";")
    .option("header", "true")
    .option("skipRows", 1)
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .option("mode", "PERMISSIVE")
    .load(CAMINHO) #carregar os arquivos com essas configs acima
)
#usando a action de read --> passando as configurações de separador, se tem cabeçalho, pra pular primeira linha, identificação de textos. Toda essa configuração é sobre como o arquivo veio. Antes de ingerir, lemos o arquivo.

print("Colunas lidas do arquivo: ")
for c in bruto.columns:
    print(f"  {c!r}")
#quando executou toda esse processo do read, criou um dataframe
#agora esses dados saíram do arquivo e estão dentro do spark para podermos fazer o que quisermos com eles pelo dataframe

In [0]:
#nome das colunas com espaço não dá, tem que corrigir. Dá pra usar regex mas vamos fazer o mapeamento
RENOMEAR = {
    "ICAO Empresa Aérea": "icao_empresa",
    "Número Voo": "numero_voo",
    "Código Autorização (DI)": "codigo_di",
    "Código Tipo Linha": "codigo_tipo_linha",
    "ICAO Aeródromo Origem": "icao_origem",
    "ICAO Aeródromo Destino": "icao_destino",
    "Partida Prevista": "partida_prevista",
    "Partida Real": "partida_real",
    "Chegada Prevista": "chegada_prevista",
    "Chegada Real": "chegada_real",
    "Situação Voo": "situacao_voo",
    "Código Justificativa": "codigo_justificativa"
}
#controle rigido de nomenclatura e coluna, garantindo que nao tenha coluna a mais e nem faltante e que tao saindo como queremos que saiam. vamos fazer o loop percorrendo todo o dataframe trazendo a substituição de cada coluna
faltando = [c for c in RENOMEAR if c not in bruto.columns]
assert not faltando, f"Coluna esperada não encontrada no arquivo CSV: {faltando}"

renomeado = bruto.select(
    *[F.col(f"`{origem}`" ).cast("string").alias(novo) for origem,
    novo in RENOMEAR.items()]
)
#todas as colunas estão tipo string pq nao tipamos na camada bronze

In [0]:
#auditoria, para garantir a rastreabilidade. withColumn no pyspark adiciona uma coluna
bronze = renomeado.withColumn(
    "_arquivo_origem", F.col("_metadata.file_name") 
).withColumn(
    "_ingerido_em", F.current_timestamp() 
)
#_arquivo_origem é o nome da coluna, pra sabermos a origem, e o dado dela é o _metadata.file_name
#_ingerido_em é para sabermos a data de ingestão disso
#vemos se tem gaps de datas, ou se executou mas tal linha nao veio, etc

In [0]:
#vamo garantir a idempotência da nossa camada, ou seja, que não estamos duplicando nenhum dado. Não quer dizer que o dado ja venha duplicado, estamos garantindo que NÓS e NOSSAS AÇÕES não estamos duplicando dados. vai ler tudo que tem dentro do volume e reescrever. Contrário de incremental, é uma ingestão completa
(
    bronze.write.format("delta") #escrita da bronze no formato delta
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(TABELA)
)
#substituindo todos os dados que colocamos na camada bronze pelos dados que tao no dataframe
print(f"{TABELA}: {spark.table(TABELA).count():,} linhas")
#como dado é bruto, nativo, nao temos opção de identificar uma alteração, temos que fazer a reescrita completa

In [0]:
spark.sql(f"""
          COMMENT ON TABLE {TABELA} IS 
          'Bronze - VRA (Voo Regular Ativo) da ANAC, 12 meses (ago/2025 a jul/2026).
           Dado bruto: todas as colunas são string, nenhuma linha descartada.
           Carga full refresh idempotente a partir de /Volumes/voebem/bronze/arquivos/vra.'
          """)
#contextualização de catálogo --> boa prática de governança. colocando dentro da tabela um contexto de pra que ela serve
#isso chama gestão de metadados: comentando sobre cada estrutura pra que serve e como se comporta. Tópico importante da governança de dados

In [0]:
display(
    spark.sql(f"""
       SELECT _arquivo_origem, COUNT(*) AS linhas, MAX
       (_ingerido_em) AS ingerido_em
       FROM {TABELA}       
       GROUP BY _arquivo_origem
       ORDER BY _arquivo_origem
    """)
)
#display printa o comando SELECT que fizemos e disponibiliza em tabela pra gente
#é uma linha para cada arquivo e quando aconteceu, pra sabermos onde deu erro e se deu erro. Se der um problema no dado, sabemos qual arquivo deu problema, pra fazer a correção de dps ingestão novamente